In [1]:
#import libraries
import psycopg2
import dash
from dash import dcc, html,Dash
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import numpy as np


In [2]:
#Reading data from postgresql

# Establish connection
conn = psycopg2.connect(
    host="localhost",
    database="APDV",
    user="dap",
    password="dap"
)

# Define query
sql_query = "SELECT * FROM evpopulation"

# Execute with parameters (prevents SQL injection)
params = ('value',)
evpopulation_data = pd.read_sql(sql_query, conn, params=params)

# Display results
print(evpopulation_data.head())

# Close connection
conn.close()

   index                 Date     County State Vehicle Primary Use  \
0      0  2018-01-31T00:00:00    Brevard    FL           Passenger   
1      1  2021-03-31T00:00:00   Pinellas    FL           Passenger   
2      2  2020-08-31T00:00:00     Shasta    CA           Passenger   
3      3  2020-06-30T00:00:00      Bucks    PA           Passenger   
4      4  2023-02-28T00:00:00  Snohomish    WA           Passenger   

  Battery Electric Vehicles (BEVs) Plug-In Hybrid Electric Vehicles (PHEVs)  \
0                                0                                        1   
1                                1                                        1   
2                                1                                        0   
3                                1                                        0   
4                            10907                                     2828   

  Electric Vehicle (EV) Total Non-Electric Vehicle Total Total Vehicles  \
0                           1

C:\Users\harig\AppData\Local\Temp\ipykernel_13124\975119667.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  evpopulation_data = pd.read_sql(sql_query, conn, params=params)


In [4]:
#shape
evpopulation_data.shape


(5073, 11)

In [5]:
#information
evpopulation_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5073 entries, 0 to 5072
Data columns (total 11 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   index                                     5073 non-null   int64 
 1   Date                                      5073 non-null   object
 2   County                                    5054 non-null   object
 3   State                                     5054 non-null   object
 4   Vehicle Primary Use                       5073 non-null   object
 5   Battery Electric Vehicles (BEVs)          5073 non-null   object
 6   Plug-In Hybrid Electric Vehicles (PHEVs)  5073 non-null   object
 7   Electric Vehicle (EV) Total               5073 non-null   object
 8   Non-Electric Vehicle Total                5073 non-null   object
 9   Total Vehicles                            5073 non-null   object
 10  Percent Electric Vehicles                 5073 n

In [6]:
#descriptive statistics
evpopulation_data.describe()

,index
count,5073.000000
mean,2536.000000
std,1464.593288
min,0.000000
25%,1268.000000
50%,2536.000000
75%,3804.000000
max,5072.000000


In [7]:
#finding missing value
evpopulation_data.isnull().sum()

index                                        0
Date                                         0
County                                      19
State                                       19
Vehicle Primary Use                          0
Battery Electric Vehicles (BEVs)             0
Plug-In Hybrid Electric Vehicles (PHEVs)     0
Electric Vehicle (EV) Total                  0
Non-Electric Vehicle Total                   0
Total Vehicles                               0
Percent Electric Vehicles                    0
dtype: int64

In [8]:
#percentage of missing value
evpopulation_data.isnull().sum()/evpopulation_data.shape[0]*100

index                                       0.000000
Date                                        0.000000
County                                      0.374532
State                                       0.374532
Vehicle Primary Use                         0.000000
Battery Electric Vehicles (BEVs)            0.000000
Plug-In Hybrid Electric Vehicles (PHEVs)    0.000000
Electric Vehicle (EV) Total                 0.000000
Non-Electric Vehicle Total                  0.000000
Total Vehicles                              0.000000
Percent Electric Vehicles                   0.000000
dtype: float64

In [9]:
#Dropping County and State column
evpopulation_data.dropna(subset=['County', 'State'], inplace=True)

In [10]:
#shape
evpopulation_data.shape

(5054, 11)

In [11]:
#Rechecking null values
evpopulation_data.isnull().sum()

index                                       0
Date                                        0
County                                      0
State                                       0
Vehicle Primary Use                         0
Battery Electric Vehicles (BEVs)            0
Plug-In Hybrid Electric Vehicles (PHEVs)    0
Electric Vehicle (EV) Total                 0
Non-Electric Vehicle Total                  0
Total Vehicles                              0
Percent Electric Vehicles                   0
dtype: int64

In [12]:
#All columns in the dataframe
evpopulation_data.columns

Index(['index', 'Date', 'County', 'State', 'Vehicle Primary Use',
       'Battery Electric Vehicles (BEVs)',
       'Plug-In Hybrid Electric Vehicles (PHEVs)',
       'Electric Vehicle (EV) Total', 'Non-Electric Vehicle Total',
       'Total Vehicles', 'Percent Electric Vehicles'],
      dtype='object')

In [13]:
# Convert to datetime and extract features
evpopulation_data['Date'] = pd.to_datetime(evpopulation_data['Date'])
evpopulation_data['Year'] = evpopulation_data['Date'].dt.year
evpopulation_data['Month'] = evpopulation_data['Date'].dt.month
evpopulation_data['Day'] = evpopulation_data['Date'].dt.day


In [14]:
evpopulation_data.head()

,index,Date,County,State,Vehicle Primary Use,Battery Electric Vehicles (BEVs),Plug-In Hybrid Electric Vehicles (PHEVs),Electric Vehicle (EV) Total,Non-Electric Vehicle Total,Total Vehicles,Percent Electric Vehicles,Year,Month,Day
0,0,2018-01-31,Brevard,FL,Passenger,0,1,1,109,110,0.9090909090909090909090909091,2018,1,31
1,1,2021-03-31,Pinellas,FL,Passenger,1,1,2,113,115,1.739130434782608695652173913,2021,3,31
2,2,2020-08-31,Shasta,CA,Passenger,1,0,1,36,37,2.702702702702702702702702703,2020,8,31
3,3,2020-06-30,Bucks,PA,Passenger,1,0,1,24,25,4.00,2020,6,30
4,4,2023-02-28,Snohomish,WA,Passenger,10907,2828,13735,528837,542572,2.531461262284083955677771798,2023,2,28


In [15]:
unique_states = pd.unique(evpopulation_data['State'])
print(unique_states)

['FL' 'CA' 'PA' 'WA' 'NH' 'HI' 'OR' 'SC' 'OH' 'MD' 'MS' 'NJ' 'ID' 'NY'
 'AL' 'KY' 'VA' 'AZ' 'UT' 'TX' 'NC' 'AK' 'MA' 'NV' 'KS' 'MO' 'CO' 'IN'
 'IL' 'GA' 'CT' 'MI' 'ME' 'NE' 'OK' 'MT' 'MN' 'TN' 'NM' 'AR' 'RI' 'DE'
 'LA' 'SD' 'DC' 'WY' 'WI' 'PR' 'ND' 'IA']


In [16]:
def add_location_features(evpopulation_data, region_mapping=None):
    """
    Adds location-based features to a DataFrame with complete US region mapping.
    
    Parameters:
    - df: pandas DataFrame containing 'County' and 'State' columns
    - region_mapping: Optional dictionary mapping states to regions
                  If None, uses complete US regions classification
    
    Returns:
    - DataFrame with added location features:
        * Location (County, State)
        * Region (4 main US regions)
        * Division (9 US census divisions)
        * Binary indicators for each region
    """
    
    # Complete US Census Bureau regions and divisions mapping
    if region_mapping is None:
        region_mapping = {
            # Northeast Region
            'CT': 'Northeast', 'ME': 'Northeast', 'MA': 'Northeast', 
            'NH': 'Northeast', 'NJ': 'Northeast', 'NY': 'Northeast',
            'PA': 'Northeast', 'RI': 'Northeast', 'VT': 'Northeast',
            
            # Midwest Region
            'IL': 'Midwest', 'IN': 'Midwest', 'IA': 'Midwest',
            'KS': 'Midwest', 'MI': 'Midwest', 'MN': 'Midwest',
            'MO': 'Midwest', 'NE': 'Midwest', 'ND': 'Midwest',
            'OH': 'Midwest', 'SD': 'Midwest', 'WI': 'Midwest',
            
            # South Region
            'AL': 'South', 'AR': 'South', 'DE': 'South',
            'FL': 'South', 'GA': 'South', 'KY': 'South',
            'LA': 'South', 'MD': 'South', 'MS': 'South',
            'NC': 'South', 'OK': 'South', 'SC': 'South',
            'TN': 'South', 'TX': 'South', 'VA': 'South',
            'WV': 'South', 'DC': 'South',
            
            # West Region
            'AK': 'West', 'AZ': 'West', 'CA': 'West',
            'CO': 'West', 'HI': 'West', 'ID': 'West',
            'MT': 'West', 'NV': 'West', 'NM': 'West',
            'OR': 'West', 'UT': 'West', 'WA': 'West',
            'WY': 'West',
            
            # Territories
            'PR': 'Territory', 'GU': 'Territory', 'VI': 'Territory'
        }
        
        # Additional division mapping (more granular)
        division_mapping = {
            # Northeast
            'CT': 'New England', 'ME': 'New England', 'MA': 'New England',
            'NH': 'New England', 'RI': 'New England', 'VT': 'New England',
            'NJ': 'Mid-Atlantic', 'NY': 'Mid-Atlantic', 'PA': 'Mid-Atlantic',
            
            # Midwest
            'IL': 'East North Central', 'IN': 'East North Central', 
            'MI': 'East North Central', 'OH': 'East North Central', 
            'WI': 'East North Central',
            'IA': 'West North Central', 'KS': 'West North Central',
            'MN': 'West North Central', 'MO': 'West North Central',
            'NE': 'West North Central', 'ND': 'West North Central',
            'SD': 'West North Central',
            
            # South
            'DE': 'South Atlantic', 'FL': 'South Atlantic', 'GA': 'South Atlantic',
            'MD': 'South Atlantic', 'NC': 'South Atlantic', 'SC': 'South Atlantic',
            'VA': 'South Atlantic', 'WV': 'South Atlantic', 'DC': 'South Atlantic',
            'AL': 'East South Central', 'KY': 'East South Central',
            'MS': 'East South Central', 'TN': 'East South Central',
            'AR': 'West South Central', 'LA': 'West South Central',
            'OK': 'West South Central', 'TX': 'West South Central',
            
            # West
            'AZ': 'Mountain', 'CO': 'Mountain', 'ID': 'Mountain',
            'MT': 'Mountain', 'NV': 'Mountain', 'NM': 'Mountain',
            'UT': 'Mountain', 'WY': 'Mountain',
            'AK': 'Pacific', 'CA': 'Pacific', 'HI': 'Pacific',
            'OR': 'Pacific', 'WA': 'Pacific'
        }
    
    # Create combined location feature
    evpopulation_data['Location'] = evpopulation_data['County'] + ', ' + evpopulation_data['State']
    
    # Add region feature
    evpopulation_data['Region'] = evpopulation_data['State'].map(region_mapping)
    
    # Add division feature if using default mapping
    if region_mapping is None:
        evpopulation_data['Division'] = evpopulation_data['State'].map(division_mapping)
    
    # Add binary indicators for regions
    for region in set(region_mapping.values()):
        evpopulation_data[f'Is_{region.lower().replace(" ", "_")}'] = (evpopulation_data['Region'] == region).astype(int)
    
    return evpopulation_data

evpopulation_data = add_location_features(evpopulation_data)

In [17]:
evpopulation_data.head()

,index,Date,County,State,Vehicle Primary Use,Battery Electric Vehicles (BEVs),Plug-In Hybrid Electric Vehicles (PHEVs),Electric Vehicle (EV) Total,Non-Electric Vehicle Total,Total Vehicles,...,Year,Month,Day,Location,Region,Is_northeast,Is_midwest,Is_south,Is_territory,Is_west
0,0,2018-01-31,Brevard,FL,Passenger,0,1,1,109,110,...,2018,1,31,"Brevard, FL",South,0,0,1,0,0
1,1,2021-03-31,Pinellas,FL,Passenger,1,1,2,113,115,...,2021,3,31,"Pinellas, FL",South,0,0,1,0,0
2,2,2020-08-31,Shasta,CA,Passenger,1,0,1,36,37,...,2020,8,31,"Shasta, CA",West,0,0,0,0,1
3,3,2020-06-30,Bucks,PA,Passenger,1,0,1,24,25,...,2020,6,30,"Bucks, PA",Northeast,1,0,0,0,0
4,4,2023-02-28,Snohomish,WA,Passenger,10907,2828,13735,528837,542572,...,2023,2,28,"Snohomish, WA",West,0,0,0,0,1


In [18]:
evpopulation_data.Year.value_counts()

Year
2024    766
2023    737
2021    679
2022    675
2020    649
2019    521
2018    485
2017    429
2025    113
Name: count, dtype: int64

In [19]:
evpopulation_data.head()

,index,Date,County,State,Vehicle Primary Use,Battery Electric Vehicles (BEVs),Plug-In Hybrid Electric Vehicles (PHEVs),Electric Vehicle (EV) Total,Non-Electric Vehicle Total,Total Vehicles,...,Year,Month,Day,Location,Region,Is_northeast,Is_midwest,Is_south,Is_territory,Is_west
0,0,2018-01-31,Brevard,FL,Passenger,0,1,1,109,110,...,2018,1,31,"Brevard, FL",South,0,0,1,0,0
1,1,2021-03-31,Pinellas,FL,Passenger,1,1,2,113,115,...,2021,3,31,"Pinellas, FL",South,0,0,1,0,0
2,2,2020-08-31,Shasta,CA,Passenger,1,0,1,36,37,...,2020,8,31,"Shasta, CA",West,0,0,0,0,1
3,3,2020-06-30,Bucks,PA,Passenger,1,0,1,24,25,...,2020,6,30,"Bucks, PA",Northeast,1,0,0,0,0
4,4,2023-02-28,Snohomish,WA,Passenger,10907,2828,13735,528837,542572,...,2023,2,28,"Snohomish, WA",West,0,0,0,0,1


In [20]:

# Convert columns to numeric, coercing errors to NaN if needed
evpopulation_data['Electric Vehicle (EV) Total'] = pd.to_numeric(evpopulation_data['Electric Vehicle (EV) Total'], errors='coerce')
evpopulation_data['Total Vehicles'] = pd.to_numeric(evpopulation_data['Total Vehicles'], errors='coerce')

# Now calculate EV_Density
evpopulation_data['EV_Density'] = evpopulation_data['Electric Vehicle (EV) Total'] / evpopulation_data['Total Vehicles']

In [21]:
# Days since first date in dataset
evpopulation_data['Days_Since_First_Record'] = (evpopulation_data['Date'] - evpopulation_data['Date'].min()).dt.days

# Quarter of year
evpopulation_data['Quarter'] = evpopulation_data['Date'].dt.quarter

In [22]:
evpopulation_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5054 entries, 0 to 5072
Data columns (total 24 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   index                                     5054 non-null   int64         
 1   Date                                      5054 non-null   datetime64[ns]
 2   County                                    5054 non-null   object        
 3   State                                     5054 non-null   object        
 4   Vehicle Primary Use                       5054 non-null   object        
 5   Battery Electric Vehicles (BEVs)          5054 non-null   object        
 6   Plug-In Hybrid Electric Vehicles (PHEVs)  5054 non-null   object        
 7   Electric Vehicle (EV) Total               5054 non-null   int64         
 8   Non-Electric Vehicle Total                5054 non-null   object        
 9   Total Vehicles                     

In [23]:
# Check if 'Percent Electric Vehicles' is numeric
print(evpopulation_data['Percent Electric Vehicles'].dtype)

# Find non-numeric values (returns rows where conversion fails)
non_numeric = evpopulation_data[pd.to_numeric(evpopulation_data['Percent Electric Vehicles'], errors='coerce').isna()]
if not non_numeric.empty:
    print("Non-numeric values found in 'Percent Electric Vehicles':")
    print(non_numeric[['County', 'State', 'Percent Electric Vehicles']].head())

object


In [24]:
non_numeric = evpopulation_data[pd.to_numeric(evpopulation_data['Percent Electric Vehicles'], errors='coerce').isna()]
print("Sample non-numeric values:")
print(non_numeric['Percent Electric Vehicles'].unique()[:10])  # Show first 10 unique bad values

Sample non-numeric values:
[]


In [25]:
# Force conversion to numeric (will succeed since there are no invalid values)
evpopulation_data['Percent Electric Vehicles'] = pd.to_numeric(
    evpopulation_data['Percent Electric Vehicles'],
    errors='coerce'  # Coerce is redundant here but safe
)

# Verify dtype is now numeric
print(evpopulation_data['Percent Electric Vehicles'].dtype)  # Should now be float64

float64


In [26]:
# For County/State based on EV adoption
county_ev_mean = evpopulation_data.groupby('County')['Percent Electric Vehicles'].mean().to_dict()
evpopulation_data['County_EV_Mean'] = evpopulation_data['County'].map(county_ev_mean)

state_ev_mean = evpopulation_data.groupby('State')['Percent Electric Vehicles'].mean().to_dict()
evpopulation_data['State_EV_Mean'] = evpopulation_data['State'].map(state_ev_mean)

In [27]:
# Bin counties by total vehicles
evpopulation_data['Vehicle_Size_Bin'] = pd.cut(evpopulation_data['Total Vehicles'],
                               bins=[0, 100, 1000, 10000, float('inf')],
                               labels=['Tiny', 'Small', 'Medium', 'Large'])

# Bin EV percentages
evpopulation_data['EV_Percent_Bin'] = pd.cut(evpopulation_data['Percent Electric Vehicles'],
                             bins=[0, 1, 2, 5, float('inf')],
                             labels=['Low', 'Medium', 'High', 'Very High'])

In [28]:
# Interaction between vehicle type and region
evpopulation_data['BEV_in_West'] = (evpopulation_data['Region'] == 'West') * evpopulation_data['Battery Electric Vehicles (BEVs)']

# Seasonality interaction
evpopulation_data['Winter_EV'] = ((evpopulation_data['Month'].isin([12, 1, 2])) * evpopulation_data['Electric Vehicle (EV) Total'])

In [29]:
# Assuming data is sorted by date
evpopulation_data['EV_Growth_Rate'] = evpopulation_data.groupby(['County', 'State'])['Electric Vehicle (EV) Total'].pct_change()
evpopulation_data['Market_Growth_Rate'] = evpopulation_data.groupby(['County', 'State'])['Total Vehicles'].pct_change()

In [ ]:
# Z-scores for outlier detection
evpopulation_data['EV_Z_Score'] = (evpopulation_data['Percent Electric Vehicles'] - evpopulation_data['Percent Electric Vehicles'].mean()) / evpopulation_data['Percent Electric Vehicles'].std()

In [31]:
import numpy as np
import pandas as pd

def clean_numeric_column(series):
    """Safely convert a pandas Series to numeric, handling various formats"""
    try:
        # First attempt direct numeric conversion
        result = pd.to_numeric(series, errors='coerce')
        
        # If that fails (many NaN values), try string cleaning
        if result.isna().mean() > 0.5:  # If more than 50% failed
            result = (
                series.astype(str)
                .str.replace(r'[^\d\.]', '', regex=True)  # Remove all non-numeric chars except .
                .replace(r'^\.$', np.nan, regex=True)  # Replace standalone . with NaN
                .replace('', np.nan)
                .astype(float)
            )
        return result
    except Exception as e:
        print(f"Error cleaning column: {e}")
        return series  # Return original if cleaning fails

# Columns to process
numeric_cols = [
    'Battery Electric Vehicles (BEVs)',
    'Plug-In Hybrid Electric Vehicles (PHEVs)',
    'Total Vehicles',
    'Electric Vehicle (EV) Total',
    'Non-Electric Vehicle Total'
]

# Clean each column
for col in numeric_cols:
    if col in evpopulation_data.columns:
        evpopulation_data[col] = clean_numeric_column(evpopulation_data[col])
    else:
        print(f"Warning: Column '{col}' not found in dataframe")

# Safe calculations with zero handling
evpopulation_data['BEV_Percentage'] = (
    evpopulation_data['Battery Electric Vehicles (BEVs)'] / 
    evpopulation_data['Total Vehicles'].replace(0, np.nan) * 100
).round(2)

evpopulation_data['PHEV_Percentage'] = (
    evpopulation_data['Plug-In Hybrid Electric Vehicles (PHEVs)'] / 
    evpopulation_data['Total Vehicles'].replace(0, np.nan) * 100
).round(2)

evpopulation_data['EV_to_NonEV_Ratio'] = (
    evpopulation_data['Electric Vehicle (EV) Total'] / 
    evpopulation_data['Non-Electric Vehicle Total'].replace(0, np.nan)
)

# Dominant EV type with tie handling
conditions = [
    evpopulation_data['Battery Electric Vehicles (BEVs)'] > evpopulation_data['Plug-In Hybrid Electric Vehicles (PHEVs)'],
    evpopulation_data['Battery Electric Vehicles (BEVs)'] < evpopulation_data['Plug-In Hybrid Electric Vehicles (PHEVs)']
]
choices = ['BEV', 'PHEV']
evpopulation_data['Dominant_EV_Type'] = np.select(conditions, choices, default='Equal')

# Display results
print(evpopulation_data[['BEV_Percentage', 'PHEV_Percentage', 'EV_to_NonEV_Ratio', 'Dominant_EV_Type']].head())

   BEV_Percentage  PHEV_Percentage  EV_to_NonEV_Ratio Dominant_EV_Type
0            0.00             0.91           0.009174             PHEV
1            0.87             0.87           0.017699            Equal
2            2.70             0.00           0.027778              BEV
3            4.00             0.00           0.041667              BEV
4            2.01             0.52           0.025972              BEV


In [32]:
# Region-technology interactions
evpopulation_data['West_BEV'] = ((evpopulation_data['Region'] == 'West') & (evpopulation_data['Dominant_EV_Type'] == 'BEV')).astype(int)





In [33]:
# Convert categoricals
cat_cols = ['County', 'State', 'Vehicle_Size_Bin', 'EV_Percent_Bin', 'Dominant_EV_Type']
evpopulation_data[cat_cols] = evpopulation_data[cat_cols].astype('category')

In [34]:
# Initialize Dash App
app = Dash(__name__)

## App Layout
app.layout = html.Div([
    html.H1("Electric Vehicle  Dashboard", style={'textAlign': 'center'}),
    
    # Filters Row
    html.Div([
        html.Div([
            html.Label("Select Year Range:"),
            dcc.RangeSlider(
                id='year-slider',
                min=evpopulation_data['Year'].min(),
                max=evpopulation_data['Year'].max(),
                value=[evpopulation_data['Year'].min(), evpopulation_data['Year'].max()],
                marks={str(year): str(year) for year in evpopulation_data['Year'].unique()},
                step=None
            )
        ], style={'width': '48%', 'display': 'inline-block'}),
        
        html.Div([
            html.Label("Select Region:"),
            dcc.Dropdown(
                id='region-dropdown',
                options=[{'label': r, 'value': r} for r in evpopulation_data['Region'].unique()],
                value=evpopulation_data['Region'].unique(),
                multi=True
            )
        ], style={'width': '48%', 'float': 'right', 'display': 'inline-block'})
    ]),
    
    # Main Charts Row
    html.Div([
        html.Div([
            dcc.Graph(id='adoption-trend')
        ], style={'width': '48%', 'display': 'inline-block'}),
        
        html.Div([
            dcc.Graph(id='regional-comparison')
        ], style={'width': '48%', 'float': 'right', 'display': 'inline-block'})
    ]),
    
    # Secondary Charts Row
    html.Div([
        html.Div([
            dcc.Graph(id='technology-mix')
        ], style={'width': '48%', 'display': 'inline-block'}),
        
        html.Div([
            dcc.Graph(id='fleet-size-impact')
        ], style={'width': '48%', 'float': 'right', 'display': 'inline-block'})
    ]),
    
    # Data Table
    html.Div([
        html.H3("Detailed Data"),
        html.Div(id='data-table')
    ])
])

@app.callback(
    [Output('adoption-trend', 'figure'),
     Output('regional-comparison', 'figure'),
     Output('technology-mix', 'figure'),
     Output('fleet-size-impact', 'figure'),
     Output('data-table', 'children')],
    [Input('year-slider', 'value'),
     Input('region-dropdown', 'value')]
)
def update_dashboard(selected_years, selected_regions):
    # Filter data
    filtered_df = evpopulation_data[
        (evpopulation_data['Year'] >= selected_years[0]) & 
        (evpopulation_data['Year'] <= selected_years[1]) & 
        (evpopulation_data['Region'].isin(selected_regions))
    ]
    
    # 1. Adoption Trend Chart
    trend_fig = px.line(
        filtered_df.groupby(['Year', 'Month'])['Percent Electric Vehicles'].mean().reset_index(),
        x='Month',
        y='Percent Electric Vehicles',
        color='Year',
        title="Monthly EV Adoption Trend",
        labels={'Percent Electric Vehicles': 'EV Adoption %'}
    )
    
    # 2. Regional Comparison Chart
    regional_fig = px.bar(
        filtered_df.groupby('Region')['Percent Electric Vehicles'].mean().reset_index(),
        x='Region',
        y='Percent Electric Vehicles',
        color='Region',
        title="Average EV Adoption by Region"
    )
    
    # 3. Technology Mix Chart
    tech_fig = px.pie(
        filtered_df.groupby('Dominant_EV_Type').size().reset_index(name='count'),
        values='count',
        names='Dominant_EV_Type',
        title="Dominant EV Technology Mix"
    )
    
    # 4. Fleet Size Impact Chart - CORRECTED
    fleet_fig = px.box(
        filtered_df,
        x='Vehicle_Size_Bin',
        y='Percent Electric Vehicles',
        color='Vehicle_Size_Bin',
        title="EV Adoption by Vehicle Size Category"
    )
    
    # 5. Data Table
    table = dash.dash_table.DataTable(
        columns=[{"name": i, "id": i} for i in filtered_df.columns],
        data=filtered_df.to_dict('records'),
        page_size=10,
        style_table={'overflowX': 'auto'}
    )
    
    return trend_fig, regional_fig, tech_fig, fleet_fig, table
if __name__ == '__main__':
    #app.run(jupyter_mode='inline', debug=True)  
    app.run(jupyter_mode='external', port=8054)# debug=True helps see errors

Dash app running on http://127.0.0.1:8054/
